In [ ]:
from typing_extensions import TypedDict, Optional, Annotated
from operator import or_

from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class State(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    user_query: str
    results: Annotated[dict, or_]
    user_genre_choice: list[str]
    user_movies_choice: list[str]

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


def scrape_moneycontrol_news(company_slug, pages=3):
    base_url = f"https://www.moneycontrol.com/news/tags/{company_slug}/"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    all_news = []

    for page in range(1, pages + 1):
        url = base_url if page == 1 else f"{base_url}page-{page}/"

        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        articles = soup.select("li.clearfix")

        print(f"Scraping page {page} - found {len(articles)} articles")

        for article in articles:
            try:
                # ✅ New selector
                title_tag = article.select_one("h2.related_des")

                if not title_tag:
                    continue  # skip login/ads blocks

                title = title_tag.text.strip()

                # ✅ URL now from parent <a>
                link_tag = article.select_one("a")
                url = link_tag.get("href") if link_tag else ""

                all_news.append({
                    "title": title,
                    "url": url
                })

            except Exception as e:
                print("ERROR:", e)

    print("Total collected:", len(all_news))

    return pd.DataFrame(all_news)

In [8]:
scrape_moneycontrol_news("reliance-industries")

Scraping page 1 - found 8 articles
Scraping page 2 - found 26 articles
Scraping page 3 - found 26 articles
Total collected: 7


,title,url
0,"Before counting Hyundai out, we need to count ...",https://www.moneycontrol.com/news/opinion/befo...
1,Can chassis deal restore Bosch’s margins to hi...,https://www.moneycontrol.com/news/business/mon...
2,Auto stocks fall up to 5% on Delhi govt's draf...,https://www.moneycontrol.com/news/business/mar...
3,‘Blockade will remain’: Trump says US won’t ea...,https://www.moneycontrol.com/world/blockade-wi...
4,Amit Shah's big commitment to Opposition: 'Giv...,https://www.moneycontrol.com/news/india/amit-s...
5,'Gulshan was once Gopal': Old photo emerges as...,https://www.moneycontrol.com/city/gulshan-was-...
6,How Rs 20 lakh salary can turn tax-free under ...,https://www.moneycontrol.com/news/business/per...


In [20]:
import feedparser
from datetime import datetime

def fetch_google_news(keyword, limit=10):
    url = f"https://news.google.com/rss/search?q={keyword}+stock&hl=en-IN&gl=IN&ceid=IN:en"

    feed = feedparser.parse(url)

    articles = []

    for entry in feed.entries[:limit]:
        articles.append({
            "title": entry.get("title"),
            "url": entry.get("link"),
            "source": "Google News",
            "published": entry.get("published"),
            "scraped_at": datetime.now().isoformat()
        })

    return articles

In [21]:
fetch_google_news("reliance")

[{'title': 'Reliance Power Share Price Rises 0.21% After Media Clarification - HDFC Sky',
  'url': 'https://news.google.com/rss/articles/CBMiqgFBVV95cUxQOG5wSzlkSm00QS1OS056a015aWoyM2RWVU5DNklMNWQwWXZfTmdVcTFRRi1zcWRYTi1vbmhfRFlXZzRZTzdUVVFZVHEwTGtqWlZud0VaekstcVlIcl9YS2RBekxzMHY4dkFrcUdPcko3X09qcGN1aTdqQm1MMEN4SkYtRWY5Y1hIWXNtNTJ5c2tVUk1rczZ0TnZoa3Y2bWNkblNzenp6UUVaUQ?oc=5',
  'source': 'Google News',
  'published': 'Fri, 17 Apr 2026 07:13:23 GMT',
  'scraped_at': '2026-04-17T19:40:17.041860'},
 {'title': 'Reliance Industries Board Meeting Scheduled for April 24, 2026 to Consider Q4FY26 Results - scanx.trade',
  'url': 'https://news.google.com/rss/articles/CBMi4AFBVV95cUxQT0dueDV5bEZuaUNmaXdRX0doUVVhMXpKVElCX2t1NDBmY1lRWlNXb2V0VlZxbnZWSGJpaWc4djRXY1NucXJVM2w4RWZuUzdGdi1BZ3RHMlpkRlR0N2taa1JldHZ0ZmxWWnowQ2drVnlyRUFacnBlb08zX1pWcm80M3ZKRzNPSmdQbDRUNGFjeG9CMmV6ZzF3a3l1MkhQREtNMTlNTk51emctVWM2TDNUSXROaXZXdU9mckY5VWMwbXlVTHRWQk5mMVVpX2gyc09TODJ2LUYtUU82ck1GMXF3NQ?oc=5',
  'source': 'Google 

In [32]:
RSS_SOURCES = {
    "Moneycontrol": "https://www.moneycontrol.com/rss/MCtopnews.xml",
    "Economic Times": "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
    "LiveMint": "https://www.livemint.com/rss/markets",
    "Business Standard": "https://www.business-standard.com/rss/markets-106.rss",
}

HEADERS = {"User-Agent": "Mozilla/5.0"}


# --- HELPERS ---
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_relevant(text, keywords):
    text = text.lower()
    return any(k.lower() in text for k in keywords)


# --- FETCH RSS ---
def fetch_rss_news(keywords, limit=20):
    articles = []

    for source, url in RSS_SOURCES.items():
        feed = feedparser.parse(url)
        print(f"Fetched {len(feed.entries)} entries from {source}")

        for entry in feed.entries[:limit]:
            title = clean_text(entry.get("title", ""))
            summary = clean_text(entry.get("summary", ""))

            if not is_relevant(title + " " + summary, keywords):
                continue

            articles.append({
                "title": title,
                "summary": summary,
                "url": entry.get("link"),
                "source": source,
                "published": entry.get("published"),
                "scraped_at": datetime.now().isoformat()
            })

    return articles


In [33]:
fetch_rss_news(["reliance", "nse", "stock", "ril", "jio"], limit=5)

Fetched 18 entries from Moneycontrol
Fetched 50 entries from Economic Times
Fetched 35 entries from LiveMint
Fetched 0 entries from Business Standard


[{'title': 'Sensex, Nifty wobbly; Hind Zinc, SBI, Force Motors most active',
  'summary': 'Asian Paints, SBI, Tata Motors, HUL and Maruti Suzuki are top gainers while ONGC, Axis Bank, MM, Bajaj Auto and Hero MotoCorp are major losers in the Sensex.',
  'url': 'http://www.moneycontrol.com/news/local-markets/sensex-nifty-wobbly-hind-zinc-sbi-force-motors-most-active_7577801.html',
  'source': 'Moneycontrol',
  'published': 'Wed, 05 Oct 2016 14:00:44 +0530',
  'scraped_at': '2026-04-17T19:51:25.475627'},
 {'title': 'Bond bull market may pause but is far from over: Expert',
  'summary': "The benchmark 10-year government-security yield remained stuck in 8-7.5 percent range through all of 2015 and half of 2016, moving lower to sub-7 percent only when the RBI promised in April to reduce the system's liquidity deficit. The yield may now fall more.",
  'url': 'http://www.moneycontrol.com/news/economy/bond-bull-market-may-pauseis-farover-expert_7575741.html',
  'source': 'Moneycontrol',
  'publi

In [35]:
import yfinance as yf

yf.Ticker("RELIANCE.NS").news

[{'id': 'c713d91e-a34e-3b38-9f2b-0837c9f68293',
  'content': {'id': 'c713d91e-a34e-3b38-9f2b-0837c9f68293',
   'contentType': 'STORY',
   'title': 'HBO Max comes to India via exclusive JioHotstar deal',
   'description': '',
   'summary': 'HBO Max will be available to JioHotstar subscribers as an add-on starting at ₹49 (about $0.50) per month and would feature content from HBO, Max Originals, Warner Bros. Pictures, Warner Bros. Television, and DC Studios.',
   'pubDate': '2026-04-15T12:47:17Z',
   'displayTime': '2026-04-15T12:47:17Z',
   'isHosted': True,
   'bypassModal': False,
   'previewUrl': None,
   'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/techcrunch_finance_785/140fdf5951f344938afa721a839960df',
    'originalWidth': 720,
    'originalHeight': 480,
    'caption': 'HBO Max icon on smartphone',
    'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/i.g0Pxw3KlGhFl8bEHtbxw--~B/aD00ODA7dz03MjA7YXBwaWQ9eXRhY2h5b24-/https://media.zenfs.com/en/techcrunch_finance_

In [36]:
import pandas as pd
import re
from datetime import datetime
import yfinance as yf


# ---------------------------
# HELPERS
# ---------------------------
def normalize_news(news_list, source_name):
    normalized = []

    for n in news_list:
        normalized.append({
            "title": n.get("title"),
            "url": n.get("url") or n.get("link"),
            "source": n.get("source", source_name),
            "published": n.get("published") or n.get("providerPublishTime"),
            "scraped_at": n.get("scraped_at", datetime.now().isoformat())
        })

    return normalized


# ---------------------------
# YFINANCE NEWS
# ---------------------------
def fetch_yfinance_news(ticker):
    stock = yf.Ticker(ticker)
    news = stock.news or []

    return [{
        "title": n.get("title"),
        "url": n.get("link"),
        "source": "Yahoo Finance",
        "published": n.get("providerPublishTime"),
        "scraped_at": datetime.now().isoformat()
    } for n in news]


# ---------------------------
# MAIN COMBINER
# ---------------------------
def build_news_dataframe(company_slug, keywords, ticker):
    
    # 1. Moneycontrol
    df_mc = scrape_moneycontrol_news(company_slug)
    
    # filter relevance
    df_mc = df_mc[df_mc["title"].str.lower().apply(
        lambda x: any(k in x for k in keywords)
    )]

    mc_news = df_mc.to_dict("records")


    # 2. RSS
    rss_news = fetch_rss_news(keywords)


    # 3. Google News
    gnews = []
    for kw in keywords:
        gnews.extend(fetch_google_news(kw))


    # 4. Yahoo Finance
    yf_news = fetch_yfinance_news(ticker)


    # ---------------------------
    # NORMALIZE
    # ---------------------------
    all_news = []
    all_news += normalize_news(mc_news, "Moneycontrol")
    all_news += normalize_news(rss_news, "RSS")
    all_news += normalize_news(gnews, "Google News")
    all_news += normalize_news(yf_news, "Yahoo Finance")


    # ---------------------------
    # CREATE DATAFRAME
    # ---------------------------
    df = pd.DataFrame(all_news)

    # drop nulls
    df = df.dropna(subset=["title", "url"])

    # ---------------------------
    # DEDUPLICATION
    # ---------------------------
    df = df.drop_duplicates(subset=["url"])

    # optional: sort by time
    df = df.sort_values(by="published", ascending=False, na_position="last")

    return df

In [37]:
df = build_news_dataframe(
    company_slug="reliance-industries",
    keywords=["reliance", "ril", "jio"],
    ticker="RELIANCE.NS"
)

print(df.head())

Page 1: 8 found
Page 2: 26 found
Page 3: 26 found
Fetched 18 entries from Moneycontrol
Fetched 50 entries from Economic Times
Fetched 35 entries from LiveMint
Fetched 0 entries from Business Standard
                                                title  \
16  Reliance Industries Ltd stock (INE002A01018): ...   
29  Dividend Stock 2026: Reliance Industries-backe...   
28  Oil-sensitive stocks rally as crude slips belo...   
15  RIL shares fall over 2% post Q1 results; analy...   
25  India Stocks Soar: Global Optimism Lifts Marke...   

                                                  url       source  \
16  https://news.google.com/rss/articles/CBMiuwFBV...  Google News   
29  https://news.google.com/rss/articles/CBMi9AFBV...  Google News   
28  https://news.google.com/rss/articles/CBMi2gFBV...  Google News   
15  https://news.google.com/rss/articles/CBMi7AFBV...  Google News   
25  https://news.google.com/rss/articles/CBMi0AFBV...  Google News   

                        published   

In [32]:
len(df)

NameError: name 'df' is not defined

In [2]:
import requests


def get_nse_filings(symbol):
    session = requests.Session()

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": "https://www.nseindia.com/",
        "Accept-Language": "en-US,en;q=0.9"
    }

    # ✅ Step 1: initialize session
    session.get("https://www.nseindia.com", headers=headers)

    # ✅ Step 2: correct endpoint
    url = f"https://www.nseindia.com/api/corporate-announcements?index=equities&symbol={symbol}"

    res = session.get(url, headers=headers)

    if res.status_code != 200:
        print("Failed:", res.status_code)
        print(res.text[:300])
        return []

    try:
        return res.json()
    except:
        print("Not JSON:")
        print(res.text[:300])
        return []

In [3]:
get_nse_filings("RELIANCE")

[{'symbol': 'RELIANCE',
  'desc': 'Analysts/Institutional Investor Meet/Con. Call Updates',
  'dt': '17042026182049',
  'attchmntFile': 'https://nsearchives.nseindia.com/corporate/kavinavora_17042026182033_SE_17042026.pdf',
  'sm_name': 'Reliance Industries Limited',
  'sm_isin': 'INE002A01018',
  'an_dt': '17-Apr-2026 18:20:49',
  'sort_date': '2026-04-17 18:20:49',
  'seq_id': '106592916',
  'smIndustry': 'Refineries',
  'orgid': None,
  'attchmntText': 'A meeting of the Board of Directors of the Company is scheduled to be held on Friday, April 24, 2026, inter alia, to consider and approve the standalone and consolidated audited financial results of the Company for the quarter and year ended March 31, 2026. The Company will hold an analyst meet, post the Board Meeting, to discuss the financial results for the quarter and year ended March 31, 2026.',
  'bflag': None,
  'old_new': None,
  'csvName': None,
  'exchdisstime': '17-Apr-2026 18:20:50',
  'difference': '00:00:01',
  'fileSize

In [4]:
import re

def classify_document(title):
    t = title.lower()

    # ---------------------------
    # 🎯 TRANSCRIPT
    # ---------------------------
    if (
        "transcript" in t or
        "earnings call" in t or
        re.search(r"q[1-4].*(call|transcript)", t)
    ):
        return "transcript"

    # ---------------------------
    # 📊 PRESENTATION
    # ---------------------------
    elif (
        "presentation" in t or
        "investor meet" in t or
        "analyst meet" in t
    ):
        return "presentation"

    # ---------------------------
    # 📈 RESULTS
    # ---------------------------
    elif (
        "result" in t or
        "financial result" in t or
        "quarter ended" in t
    ):
        return "results"

    # ---------------------------
    # ❓ OTHER
    # ---------------------------
    return "other"

In [5]:
def extract_documents(data):
    docs = {
        "transcript": [],
        "presentation": [],
        "results": []
    }

    for item in data:
        title = item.get("desc", "")
        url = item.get("attchmntFile")

        if not url:
            continue

        doc_type = classify_document(title)

        if doc_type in docs:
            docs[doc_type].append({
                "title": title,
                "url": url
            })

    return docs

In [14]:
data = get_nse_filings("TCS")

docs = extract_documents(data)

print("Transcripts:", len(docs["transcript"]))
print("Presentations:", len(docs["presentation"]))
print("Results:", len(docs["results"]))

Transcripts: 3
Presentations: 166
Results: 132


In [15]:
docs["transcript"][1]

{'title': 'Transcript of Analysts/Institutional Investor Meet/Con. Call',
 'url': 'https://nsearchives.nseindia.com/corporate/TCS_13072022184915_SEinttranscript13072022.pdf'}

In [11]:
!pip install pdfplumber

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached cryptography-46.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached cryptography-46.0.7-cp311-abi3-win_amd64.whl (3.5 MB)
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   -------------- ------------------------- 1.3/3.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 11.7 MB/s  0:00:00

   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ----------------

In [30]:
import pdfplumber


def extract_pdf_text(url):
    import requests
    import tempfile

    res = requests.get(url)

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as f:
        f.write(res.content)
        path = f.name

    text = ""

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""

    return text

In [ ]:
extract_pdf_text(docs["presentation"][2]["url"])